# Импорты

In [9]:
from collections.abc import Callable
from typing import List

import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Математическая модель

Уравнение: <br>
$
\begin{cases}
    \displaystyle V' = \frac{3V}{t - 75} \\
    V(0) = 6
\end{cases}
$

Решение: <br>
$\displaystyle
V(t) = 6 \left| \frac{t}{75} - 1 \right|^3
$

In [10]:
t_0 = 0
t_n = 60

V_0 = 6

def f(t: float, V: float) -> float:
    return 3 * V / (t - 75)

def V(t: float) -> float:
    return 6 * np.abs(t / 75 - 1) ** 3

# Решение

Решим уравнение методом Рунге-Кутта 4 порядка, но в обратном направлении:<br>
$\displaystyle
V_{i + 1} = V_i + \frac{h}{6} (k_1 + 2k_2 + 2k_3 + k_4), \\
k_1 = f(t_i, V_i), \\
k_2 = f \left( t_i + \frac{h}{2}, V_i + \frac{h}{2} k_1 \right), \\
k_3 = f \left( t_i + \frac{h}{2}, V_i + \frac{h}{2} k_2 \right), \\
k_4 = f(t_i + h, V_i + h k_3).
$

In [11]:
def runge_kutta_4(n: int, f: Callable[[float, float], float], t: List[float]) -> List[float]:
    h = (t_n - t_0) / n
    y_numeric = [V_0]

    for i in range(n):
        k_1 = f(t[i], y_numeric[i])
        k_2 = f(t[i] + h / 2, y_numeric[i] + h / 2 * k_1)
        k_3 = f(t[i] + h / 2, y_numeric[i] + h / 2 * k_2)
        k_4 = f(t[i] + h, y_numeric[i] + h * k_3)

        y_numeric.append(y_numeric[i] + h / 6 * (k_1 + 2 * k_2 + 2 * k_3 + k_4))

    return y_numeric

# Графики

In [12]:
def compare_solutions(n: int, f: Callable[[float, float], float], V: Callable[[float], float]):
    t = np.linspace(t_0, t_n, n + 1)

    V_exact = np.array([V(x_i) for x_i in t])
    V_numeric = np.array(runge_kutta_4(n, f, t))
    error = np.abs(V_exact - V_numeric)

    figure = make_subplots(rows=1, cols=2)

    figure.add_trace(
        go.Scatter(x=t, y=V_exact, mode="lines", name="Точное", legend="legend"),
        row=1, col=1,
    )
    figure.add_trace(
        go.Scatter(x=t, y=V_numeric, mode="lines", name="Численное", legend="legend"),
        row=1, col=1,
    )

    figure.add_trace(
        go.Scatter(x=t, y=error, mode="lines", name="Погрешность", legend="legend2"),
        row=1, col=2,
    )

    figure.update_yaxes(exponentformat="power", row=1, col=1)
    figure.update_yaxes(exponentformat="power", row=1, col=2)
    figure.update_xaxes(title_text="t", row=1, col=1)
    figure.update_xaxes(title_text="t", row=1, col=2)
    figure.update_yaxes(title_text="V", row=1, col=1)
    figure.update_yaxes(title_text="error", row=1, col=2)

    figure.update_layout(
        height=600,
        legend=dict(
            orientation="v", 
            xanchor="right", 
            yanchor="top", 
            x=0.45,
        ),
        legend2=dict(
            orientation="v", 
            xanchor="right", 
            yanchor="top", 
            x=1,
        ),
    )
    figure.show()

ns = [10, 100, 1000]
for n in ns:
    compare_solutions(n, f, V)